# noise-batch-from-latent — ex2: device-targeted noise with optional unit-sphere normalization

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `noise-batch-from-latent`. Running the final beacon cell reports progress against the `GAN: Noise batch from latent_dim` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: Noise batch from latent_dim` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`noise-batch-from-latent`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "noise-batch-from-latent"
DD_SUBTOPIC = "GAN: Noise batch from latent_dim"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Noise batch on a target device + unit-sphere normalization

Ex1 built `(B, L, 1, 1)` standard-normal noise. Two production extensions:

1. **Build directly on a target device** — `device=` kwarg saves a later `.to(device)` (one host→GPU copy, often the slowest step).
2. **Unit-sphere normalization** — divide each per-sample vector by its L2 norm so latent codes lie on the unit hypersphere. Used in StyleGAN, BigGAN, and any setup where you want bounded latent interpolation:

```python
noise = t.randn(B, L, 1, 1, device=device, generator=g)
norms = noise.flatten(1).norm(dim=1)         # (B,)
noise = noise / norms.view(B, 1, 1, 1).clamp_min(1e-8)
```

**Why clamp the norm.** Theoretically the norm of `N(0, I_L)` is never zero, but at low `L` you can hit very small norms numerically. The `clamp_min(1e-8)` makes the divide safe even with `L=2`.

**Sphere prior vs Gaussian prior.** Both train; the sphere prior has smoother interpolations (no probability mass near origin) but tighter support — your G must adapt.

### Exercise 2 — device-targeted noise with optional unit-sphere normalization

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `t.randn(B, L, 1, 1, device=device, generator=g)` followed by optional per-sample L2 normalization (with `clamp_min(1e-8)` safety) to project latent codes onto the unit hypersphere.
> Keywords: dcgan, noise, device, sphere-normalize, L2-norm
> ```

**KCs targeted:** `randn-with-device-kwarg`, `per-sample-l2-normalize-to-unit-sphere`

Implement `ex2_dcgan_noise(batch_size, latent_dim, generator, device, normalize=False)`. Two-mode noise builder:

1. Build standard-normal noise of shape `(batch_size, latent_dim, 1, 1)` directly on `device` (use the `device=` kwarg to `t.randn`, NOT a post-hoc `.to(device)`).
2. Pass the `generator` kwarg through for reproducibility.
3. If `normalize=True`:
   - Compute per-sample L2 norms over the last 3 dims (latent + spatial): `norms = noise.flatten(1).norm(dim=1)` — shape `(B,)`.
   - Divide each sample by its norm, clamping the divisor below by `1e-8` for safety: `noise = noise / norms.view(B, 1, 1, 1).clamp_min(1e-8)`.
4. Return the (possibly normalized) noise tensor.

Important shape contract:
- Output shape ALWAYS `(B, L, 1, 1)`, with or without normalize.
- Output device ALWAYS `device`.
- When `normalize=True`, each sample's L2 norm is `1.0 ± 1e-5`.

Input: `batch_size`, `latent_dim` — ints; `generator` — `t.Generator`; `device` — `t.device` or device-string; `normalize` — bool, default False.
Output: `(B, L, 1, 1)` float32 tensor on `device`.

The visualization plots per-sample L2 norm histograms for both modes side by side — the unnormalized form is `χ`-distributed around √latent_dim, the normalized form is a delta at 1.0.

In [ ]:
def ex2_dcgan_noise(batch_size: int, latent_dim: int, generator: 'torch.Generator',
                    device, normalize: bool = False) -> Tensor:
    noise = t.randn(batch_size, latent_dim, 1, 1, device=device, generator=generator)
    if normalize:
        norms = noise.flatten(1).norm(dim=1)
        noise = noise / norms.view(batch_size, 1, 1, 1).clamp_min(1e-8)
    return noise


<details><summary>Solution</summary>

```python
def ex2_dcgan_noise(batch_size: int, latent_dim: int, generator: 'torch.Generator',
                    device, normalize: bool = False) -> Tensor:
    noise = t.randn(batch_size, latent_dim, 1, 1, device=device, generator=generator)
    if normalize:
        norms = noise.flatten(1).norm(dim=1)
        noise = noise / norms.view(batch_size, 1, 1, 1).clamp_min(1e-8)
    return noise
```

**`device=` in `t.randn` avoids a host→device copy.** Building on CPU then `.to('cuda')` does two allocations and one PCIe copy. Passing `device=` builds the tensor on the target device directly — a meaningful speedup when sampling thousands of batches in a real training loop.

**Why per-sample (not whole-batch) normalization.** Each sample is an independent latent code; the unit sphere lives in `latent_dim`-space. Normalizing the entire batch tensor as one vector would couple samples together — nonsense.

**`flatten(1)` then `norm(dim=1)`.** Collapses the L + spatial dims into a single feature axis per sample, then computes L2 along it. Returns shape `(B,)`. The `view(B, 1, 1, 1)` broadcasts the per-sample scalar back across the original shape.

**Why `clamp_min(1e-8)`, not `clamp(min=1e-8)`.** Same call — `clamp_min` is the one-arg form. Either works; `clamp_min` reads tighter.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()